# Beijing Air Quality Forecasting

This notebook demonstrates OTP-FM for modeling temporal evolution of PM2.5 concentrations in Beijing.

The dataset contains 1D PM2.5 measurements from the Dingling station across 13 months.

In [ ]:
import torch
import numpy as np
from collections import OrderedDict
from pathlib import Path

# Import OTP-FM
from otpfm import OTPFM
from otpfm.potentials import W2InfPotential

# Import experiment utilities
from experiments.beijingair.data import load_beijing_data, create_beijing_dataloaders
from experiments.beijingair import BeijingTrainer

## 1. Load Data

In [ ]:
# Load data (downloads automatically)
data = load_beijing_data(
    data_dir=Path("data/beijing"),
    station="Dingling",
    normalize=True,
    ot_coupling=True,
)

print(f"Number of time points: {len(data['marginals'])}")
print(f"Training times: {data['train_times']}")
print(f"Holdout times: {data['holdout_times']}")
for t, m in data['marginals'].items():
    print(f"  Time {t}: {len(m)} measurements")

## 2. Create Model and Train

In [ ]:
# Create dataloaders
train_loader, val_loader = create_beijing_dataloaders(
    marginals=data['marginals_list'],
    batch_size=128,
    holdout_times=data['holdout_times'],
    ot_alignments=data['ot_alignments'],
)

# Define intermediate times
train_times = data['train_times']
tks = [(t - min(train_times)) / (max(train_times) - min(train_times)) 
       for t in train_times[1:-1]]

# Create potentials
potentials = OrderedDict()
for tk in tks:
    potentials[tk] = W2InfPotential(
        tk=tk,
        strength=100.0,
        lambda_type='gaussian',
        width=0.2,
    )

# Create model (1D data)
model = OTPFM(
    d=1,
    tks=tks,
    potentials=potentials,
    flownet_args={
        'hidden_dim': 64,
        'num_hidden_layers': 2,
    }
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train
trainer = BeijingTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir=Path("runs/beijing_demo"),
    lr=1e-3,
    epochs=100,
    potentials=potentials,
    marginals=data['marginals'],
    train_times=data['train_times'],
    holdout_times=data['holdout_times'],
    scaler=data['scaler'],
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

trainer.train()

## Reproduce Paper Results

```bash
python experiments/train.py --dataset beijingair --potential {W2, W2Inf, KL, MMD}
```